In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

# Load data
tropomi_combined_dropna = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/Run_20250623_203825/updated_tropomi_hourly_emissions_full_variables.csv').dropna()

# Get feature names
FEATURES = [
    'surface_altitude', 'surface_altitude_precision',
    'surface_classification', 'surface_pressure', 'surface_albedo',
    'surface_albedo_nitrogendioxide_window', 'cloud_pressure_crb',
    'cloud_fraction_crb', 'cloud_albedo_crb', 'scene_albedo',
    'apparent_scene_pressure', 'snow_ice_flag', 'aerosol_index_354_388', 
    'scaled_small_pixel_variance', 
    'sensor_altitude', 'sensor_azimuth_angle', 'sensor_zenith_angle', 
    'solar_azimuth_angle', 'solar_zenith_angle', 'wind_speed', 't2m', 'tisr', 'tcwv',
    'primary_fuel_type', 'NOx Mass (lbs)'
]

feature_names = FEATURES
le = LabelEncoder()
tropomi_combined_dropna['primary_fuel_type'] = le.fit_transform(tropomi_combined_dropna['primary_fuel_type'])
X = tropomi_combined_dropna[FEATURES].values.astype(np.float32)
y = tropomi_combined_dropna["plume_label"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ros = RandomOverSampler(random_state=42)
X_tr_bal, y_tr_bal = ros.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_bal)
X_te_s = scaler.transform(X_test)

# Model definition
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

# Load trained model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(X_tr_s.shape[1]).to(device)
model.load_state_dict(torch.load('/net/fs06/d3/rzhuang/TROPOMI_US/model/best_model_hourly_all_data_filtered_yearly_no_stats.pt'))
model.eval()

# Calculate original AUC
test_ds = TensorDataset(torch.from_numpy(X_te_s), torch.from_numpy(y_test))
test_dl = DataLoader(test_ds, batch_size=32)

def calculate_auc(model, data_loader, y_true):
    probs = []
    with torch.no_grad():
        for xb, _ in data_loader:
            xb = xb.to(device)
            probs.extend(torch.sigmoid(model(xb)).cpu().numpy())
    return roc_auc_score(y_true, probs)

orig_auc = calculate_auc(model, test_dl, y_test)
print(f"Original AUC: {orig_auc:.4f}")

# Permutation importance
n_features = X_te_s.shape[1]
importances = np.zeros(n_features)
n_repeats = 5  # Number of times to repeat permutation

print("\nCalculating permutation importance...")
for i in tqdm(range(n_features)):
    scores = []
    for _ in range(n_repeats):
        # Create a copy of test data
        X_perm = X_te_s.copy()
        # Permute feature i
        np.random.shuffle(X_perm[:, i])
        # Create new dataloader with permuted data
        perm_ds = TensorDataset(torch.from_numpy(X_perm), torch.from_numpy(y_test))
        perm_dl = DataLoader(perm_ds, batch_size=32)
        # Calculate AUC with permuted feature
        perm_auc = calculate_auc(model, perm_dl, y_test)
        # Importance is the drop in AUC
        scores.append(orig_auc - perm_auc)
    importances[i] = np.mean(scores)

# Create results dataframe
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(importance_df.head(20))

# Save results
importance_df.to_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/Run_20250623_203825/feature_importance_permutation.csv', index=False)

# Visualize
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
top_features = importance_df.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Permutation Importance (AUC drop)')
plt.title('Top 20 Most Important Features')
plt.tight_layout()
plt.savefig('/net/fs06/d3/rzhuang/TROPOMI_US/data/Run_20250623_203825/feature_importance_plot.png', dpi=300)
plt.close()

print(f"\nResults saved to feature_importance_permutation.csv")
print(f"Plot saved to feature_importance_plot.png")

Original AUC: 0.7881

Calculating permutation importance...


100%|██████████| 25/25 [12:20<00:00, 29.63s/it]



Top 20 Most Important Features:
                                  feature  importance
24                         NOx Mass (lbs)    0.184912
19                             wind_speed    0.024966
16                    sensor_zenith_angle    0.017928
0                        surface_altitude    0.013490
23                      primary_fuel_type    0.007902
14                        sensor_altitude    0.005378
22                                   tcwv    0.003945
5   surface_albedo_nitrogendioxide_window    0.003681
3                        surface_pressure    0.003601
15                   sensor_azimuth_angle    0.003307
20                                    t2m    0.003269
21                                   tisr    0.002898
11                          snow_ice_flag    0.002229
18                     solar_zenith_angle    0.002203
13            scaled_small_pixel_variance    0.001985
6                      cloud_pressure_crb    0.001713
12                  aerosol_index_354_388    0.00

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Permutation Importance with Year-by-Year Plant/City Interference Filtering

What this does
--------------
1) Loads plant metadata, annual emissions, and world cities.
2) Extracts year from utc_time column in TROPOMI data.
3) Identifies *interfered* plants BY YEAR using geometric & emissions logic
   (BallTree + haversine; plant radius + population-scaled city radius).
4) Builds two TROPOMI datasets:
     - unfiltered_all: all rows for plants present in the TROPOMI CSV
     - filtered_all: rows where plant 'location' NOT in interference zone for that specific year
5) For each dataset, evaluates a trained MLP and computes permutation importance
   (AUC drop when permuting each feature).
6) Saves CSVs and plots under /no_interference_yearly/{unfiltered,filtered}/

Notes
-----
- Year-specific filtering accounts for temporal changes in plant emissions.
- Uses annual emissions data to determine interference zones per year.
- Requires: scikit-learn, imbalanced-learn, haversine, joblib, torch.
"""

import os
import time
import json
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import roc_auc_score, average_precision_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.neighbors import BallTree
from haversine import haversine
from math import radians, log10

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# -------------------------
# Paths / Config
# -------------------------
DATA_TROPOMI = '/net/fs06/d3/rzhuang/TROPOMI_US/data/Run_20250623_203825/updated_tropomi_hourly_emissions_full_variables.csv'
DATA_PLANTS  = '/net/fs06/d3/rzhuang/TROPOMI_US/data/facility_emissions_by_plant_comprehensive.csv'
DATA_ANNUAL  = '/net/fs06/d3/rzhuang/TROPOMI_US/data/annual-emissions-facility-aggregation-2019-2024.csv'
DATA_CITIES  = '/net/fs06/d3/rzhuang/TROPOMI_world/data/worldcities.csv'
MODEL_PATH   = '/net/fs06/d3/rzhuang/TROPOMI_US/model/best_model_hourly_all_data.pt'
MODEL_PATH_FILTERED = '/net/fs06/d3/rzhuang/TROPOMI_US/model/best_model_hourly_all_data_filtered_yearly_no_stats.pt'
OUT_ROOT     = '/net/fs06/d3/rzhuang/TROPOMI_US/data/Run_20250623_203825/no_interference_yearly'

os.makedirs(OUT_ROOT, exist_ok=True)

BATCH_SIZE     = 2048
TEST_SIZE      = 0.20
RANDOM_STATE   = 42
N_REPEATS      = 5
NUM_WORKERS    = 0
PIN_MEMORY     = torch.cuda.is_available()
TOPK_PLOT      = 20

# Interference constants
EARTH_RADIUS_KM     = 6371
PLANT_RADIUS_BASE_KM = 20.0
CITY_POP_THRESHOLD   = 200000
CITY_RADIUS_SCALE    = 9.0
CITY_RADIUS_BASE_KM  = 10.0
CITY_RADIUS_MAX_KM   = 90.0

# Features (must match model)
FEATURES: List[str] = [
    'surface_altitude', 'surface_altitude_precision',
    'surface_classification', 'surface_pressure', 'surface_albedo',
    'surface_albedo_nitrogendioxide_window', 'cloud_pressure_crb',
    'cloud_fraction_crb', 'cloud_albedo_crb', 'scene_albedo',
    'apparent_scene_pressure', 'snow_ice_flag', 'aerosol_index_354_388',
    'scaled_small_pixel_variance',
    'sensor_altitude', 'sensor_azimuth_angle', 'sensor_zenith_angle',
    'solar_azimuth_angle', 'solar_zenith_angle', 'wind_speed', 't2m', 'tisr', 'tcwv',
    'primary_fuel_type', 'NOx Mass (lbs)'
]

# -------------------------
# Logging & utils
# -------------------------
def log(msg: str) -> None:
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def safe_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    try:
        if len(np.unique(y_true)) < 2:
            return float('nan')
        return float(roc_auc_score(y_true, y_prob))
    except Exception:
        return float('nan')

# -------------------------
# Year extraction
# -------------------------
def extract_year_from_utc_time(df: pd.DataFrame) -> pd.DataFrame:
    """Extract year from utc_time column in TROPOMI data."""
    if 'year' in df.columns:
        log(f"Found 'year' column directly. Years present: {sorted(df['year'].unique())}")
        return df
    
    if 'utc_time' in df.columns:
        try:
            # Try different parsing approaches
            try:
                df['year'] = pd.to_datetime(df['utc_time'], utc=True).dt.year
            except:
                df['year'] = pd.to_datetime(df['utc_time']).dt.year
        except Exception as e:
            log(f"Error parsing utc_time: {e}")
            # Fallback: extract year using string operations
            df['year'] = df['utc_time'].astype(str).str[:4].astype(int)
        
        years_in_data = sorted(df['year'].unique())
        log(f"Successfully extracted years: {years_in_data}")
        log(f"Year distribution in TROPOMI data:")
        for year in years_in_data:
            count = len(df[df['year'] == year])
            log(f"  Year {year}: {count:,} observations")
    else:
        log("ERROR: 'utc_time' column not found! Defaulting to 2024")
        df['year'] = 2024
    
    return df

# -------------------------
# Model
# -------------------------
class MLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)

def load_model(input_dim: int, path: str, device: torch.device) -> nn.Module:
    model = MLP(input_dim).to(device)
    state = torch.load(path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    return model

def predict_probs(model: nn.Module, X: np.ndarray, batch_size: int = BATCH_SIZE) -> np.ndarray:
    ds = TensorDataset(torch.from_numpy(X))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    probs = np.empty((len(X),), dtype=np.float32)
    i0 = 0
    with torch.inference_mode():
        for (xb,) in dl:
            xb = xb.to(next(model.parameters()).device, non_blocking=True, dtype=torch.float32)
            p = torch.sigmoid(model(xb)).detach().cpu().numpy().reshape(-1)
            n = len(p)
            probs[i0:i0+n] = p
            i0 += n
    return probs

# -------------------------
# Year-by-Year Interference filtering
# -------------------------
def identify_plants_in_interference_zones_by_year(plants_df: pd.DataFrame,
                                                  annual_emissions_df: pd.DataFrame,
                                                  cities_df: pd.DataFrame,
                                                  plant_subset_ids: Optional[List[Any]] = None) -> Dict[int, set]:
    """
    Identifies plants in interference zones for each year.
    Returns a dictionary mapping year to set of interfered plant IDs.
    """
    log("Identifying plants in interference zones by year...")
    
    # Filter to subset if provided
    if plant_subset_ids is not None:
        plants_df = plants_df[plants_df['Facility_ID'].isin(plant_subset_ids)].copy()
        annual_emissions_df = annual_emissions_df[annual_emissions_df['Facility ID'].isin(plant_subset_ids)].copy()
    
    # Merge plant locations with annual emissions
    plants_df_for_merge = plants_df.copy()
    if 'Facility_ID' in plants_df_for_merge.columns:
        plants_df_for_merge.rename(columns={'Facility_ID': 'Facility ID'}, inplace=True)
    
    # Drop conflicting columns before merge
    columns_to_drop = ['Latitude', 'Longitude', 'State', 'Facility_Name', 'Primary_Fuel_Type']
    plants_df_for_merge = plants_df_for_merge.drop(columns=columns_to_drop, errors='ignore')
    
    full_plant_data = pd.merge(annual_emissions_df, plants_df_for_merge, on='Facility ID', how='left')
    full_plant_data.dropna(subset=['Latitude', 'Longitude'], inplace=True)
    
    # Build city tree
    cities_filtered = cities_df[cities_df['population'] >= CITY_POP_THRESHOLD].copy()
    cities_filtered['lat_rad'] = np.radians(cities_filtered['latitude'])
    cities_filtered['lon_rad'] = np.radians(cities_filtered['longitude'])
    
    if len(cities_filtered) > 0:
        city_tree = BallTree(cities_filtered[['lat_rad', 'lon_rad']].values, metric='haversine')
    else:
        city_tree = None
        log("Warning: No cities above population threshold")
    
    # Process each year
    years = sorted(full_plant_data['Year'].unique())
    year_interfered_plants = {}
    
    for year in years:
        log(f"  Processing year {year}...")
        plants_this_year = full_plant_data[full_plant_data['Year'] == year].copy()
        
        if plants_this_year.empty:
            year_interfered_plants[year] = set()
            continue
        
        # Build plant tree for this year
        plants_this_year['lat_rad'] = np.radians(plants_this_year['Latitude'])
        plants_this_year['lon_rad'] = np.radians(plants_this_year['Longitude'])
        plant_tree = BallTree(plants_this_year[['lat_rad', 'lon_rad']].values, metric='haversine')
        
        interfered_plants = set()
        
        for idx, target_plant in plants_this_year.iterrows():
            target_id = target_plant['Facility ID']
            target_lat = target_plant['Latitude']
            target_lon = target_plant['Longitude']
            target_emissions = target_plant.get('NOx Mass (short tons)', 0)
            
            if pd.isna(target_lat) or pd.isna(target_lon):
                continue
                
            target_coords_rad = np.array([[radians(target_lat), radians(target_lon)]])
            
            # Check interference from other plants
            search_radius = PLANT_RADIUS_BASE_KM / EARTH_RADIUS_KM
            nearby_plant_indices = plant_tree.query_radius(target_coords_rad, r=search_radius)[0]
            
            for plant_idx in nearby_plant_indices:
                source_plant = plants_this_year.iloc[plant_idx]
                if source_plant['Facility ID'] == target_id:
                    continue
                
                source_emissions = source_plant.get('NOx Mass (short tons)', 0)
                
                # Only interfered if source has higher emissions
                if pd.notna(source_emissions) and source_emissions > target_emissions:
                    distance_km = haversine(
                        (target_lat, target_lon),
                        (source_plant['Latitude'], source_plant['Longitude'])
                    )
                    if distance_km < PLANT_RADIUS_BASE_KM:
                        interfered_plants.add(target_id)
                        break
            
            # Check interference from cities
            if city_tree and target_id not in interfered_plants:
                search_radius = CITY_RADIUS_MAX_KM / EARTH_RADIUS_KM
                nearby_city_indices = city_tree.query_radius(target_coords_rad, r=search_radius)[0]
                
                for city_idx in nearby_city_indices:
                    source_city = cities_filtered.iloc[city_idx]
                    population = source_city['population']
                    
                    # Calculate city interference radius
                    radius = CITY_RADIUS_BASE_KM + (log10(max(1, population)) * CITY_RADIUS_SCALE)
                    interference_radius_km = min(radius, CITY_RADIUS_MAX_KM)
                    
                    distance_km = haversine(
                        (target_lat, target_lon),
                        (source_city['latitude'], source_city['longitude'])
                    )
                    
                    if distance_km < interference_radius_km:
                        interfered_plants.add(target_id)
                        break
        
        year_interfered_plants[year] = interfered_plants
        log(f"    Year {year}: {len(interfered_plants)} plants in interference zones out of {len(plants_this_year)}")
    
    return year_interfered_plants

def filter_data_by_year_interference(tropomi_df: pd.DataFrame, 
                                     year_interfered_dict: Dict[int, set]) -> pd.DataFrame:
    """Filter TROPOMI data to exclude observations from plants in interference zones for their respective year."""
    filtered_dfs = []
    
    # Check what years are in the data
    years_in_data = sorted(tropomi_df['year'].unique())
    log(f"Filtering data by year-specific interference zones...")
    
    total_removed = 0
    for year in years_in_data:
        year_data = tropomi_df[tropomi_df['year'] == year].copy()
        
        if year in year_interfered_dict:
            interfered_ids = year_interfered_dict[year]
            # Keep only plants NOT in interference zones for this year
            year_data_filtered = year_data[~year_data['location'].isin(interfered_ids)]
            filtered_dfs.append(year_data_filtered)
            removed = len(year_data) - len(year_data_filtered)
            total_removed += removed
            log(f"  Year {year}: {len(year_data):,} -> {len(year_data_filtered):,} observations (removed {removed:,})")
        else:
            # If no interference data for this year, keep all observations
            filtered_dfs.append(year_data)
            log(f"  Year {year}: {len(year_data):,} observations (no interference data)")
    
    log(f"Total observations removed: {total_removed:,}")
    
    if filtered_dfs:
        return pd.concat(filtered_dfs, ignore_index=True)
    else:
        log("WARNING: No data remaining after filtering!")
        return pd.DataFrame()

# -------------------------
# Permutation importance runner
# -------------------------
def run_perm_importance_for_dataset(dataset_name: str,
                                    df: pd.DataFrame,
                                    model_path: str,
                                    out_dir: str,
                                    device: torch.device) -> Dict[str, Any]:
    """
    Preprocess (oversample+scale), load model, compute baseline metrics and permutation importance.
    Saves CSV+PNG under out_dir. Returns a summary dict.
    """
    os.makedirs(out_dir, exist_ok=True)

    # Ensure features exist and encode fuel if needed
    missing = [f for f in FEATURES if f not in df.columns]
    if missing:
        raise KeyError(f"[{dataset_name}] Missing features: {missing}")

    # Label encode primary_fuel_type if not numeric
    if not pd.api.types.is_integer_dtype(df['primary_fuel_type']) and not pd.api.types.is_float_dtype(df['primary_fuel_type']):
        le = LabelEncoder()
        df = df.copy()
        df['primary_fuel_type'] = le.fit_transform(df['primary_fuel_type'])

    # Drop NA only on required columns
    need_cols = set(FEATURES) | {'plume_label'}
    dfx = df.dropna(subset=list(need_cols)).copy()

    if len(dfx) == 0:
        log(f"[{dataset_name}] Empty after NA drop. Skipping.")
        return {"n": 0}

    X = dfx[FEATURES].values.astype(np.float32)
    y = dfx['plume_label'].astype(int).values

    # Stratified split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    # Oversample train
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_trb, y_trb = ros.fit_resample(X_tr, y_tr)

    # Scale
    scaler = StandardScaler()
    X_trs = scaler.fit_transform(X_trb)
    X_tes = scaler.transform(X_te)

    # Load model
    model = load_model(input_dim=X_trs.shape[1], path=model_path, device=device)

    # Baseline metrics
    y_prob = predict_probs(model, X_tes, batch_size=BATCH_SIZE)
    auc = safe_roc_auc(y_te, y_prob)
    pr_auc = float(average_precision_score(y_te, y_prob))
    log(f"[{dataset_name}] Holdout: ROC-AUC={auc:.4f} | PR-AUC={pr_auc:.4f} | N={len(y_te)}")

    # Permutation importance
    n_features = X_tes.shape[1]
    importances = np.zeros(n_features, dtype=np.float32)
    rng = np.random.default_rng(RANDOM_STATE)

    log(f"[{dataset_name}] Computing permutation importance (AUC drop) for {n_features} features, repeats={N_REPEATS}...")
    for i in range(n_features):
        drops = []
        for _ in range(N_REPEATS):
            X_perm = X_tes.copy()
            rng.shuffle(X_perm[:, i])
            p = predict_probs(model, X_perm, batch_size=BATCH_SIZE)
            perm_auc = safe_roc_auc(y_te, p)
            drops.append(auc - perm_auc)
        importances[i] = float(np.mean(drops))

    # Save CSV
    imp_df = pd.DataFrame({
        'feature': FEATURES,
        'importance_auc_drop': importances
    }).sort_values('importance_auc_drop', ascending=False).reset_index(drop=True)

    csv_path = os.path.join(out_dir, f'feature_importance_permutation_{dataset_name}.csv')
    imp_df.to_csv(csv_path, index=False)
    log(f"[{dataset_name}] Saved CSV: {csv_path}")

    # Plot top-k
    top_df = imp_df.head(TOPK_PLOT).iloc[::-1]
    plt.figure(figsize=(10, 8))
    plt.barh(range(len(top_df)), top_df['importance_auc_drop'])
    plt.yticks(range(len(top_df)), top_df['feature'])
    plt.xlabel('Permutation Importance (AUC drop)')
    plt.title(f'Top {TOPK_PLOT} Features — {dataset_name}')
    plt.tight_layout()
    png_path = os.path.join(out_dir, f'feature_importance_permutation_top{TOPK_PLOT}_{dataset_name}.png')
    plt.savefig(png_path, dpi=300)
    plt.close()
    log(f"[{dataset_name}] Saved plot: {png_path}")

    return {
        "n_rows_after_na": int(len(dfx)),
        "n_holdout": int(len(y_te)),
        "auc": float(auc),
        "pr_auc": float(pr_auc),
        "csv": csv_path,
        "png": png_path,
        "top_5_features": imp_df.head(5)['feature'].tolist(),
        "top_5_importances": imp_df.head(5)['importance_auc_drop'].tolist()
    }

# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    set_seed(RANDOM_STATE)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load metadata
    log(f"Loading plants: {DATA_PLANTS}")
    plants = pd.read_csv(DATA_PLANTS)
    if 'NOx_Rank' in plants.columns:
        plants = plants.sort_values(by='NOx_Rank', ascending=True)

    log(f"Loading annual emissions: {DATA_ANNUAL}")
    annual_emissions = pd.read_csv(DATA_ANNUAL)

    log(f"Loading cities: {DATA_CITIES}")
    cities = pd.read_csv(DATA_CITIES)

    # Load TROPOMI (raw)
    log(f"Loading TROPOMI: {DATA_TROPOMI}")
    tropomi_raw = pd.read_csv(DATA_TROPOMI)

    # Extract year from TROPOMI data
    log("Extracting year from TROPOMI data...")
    tropomi_raw = extract_year_from_utc_time(tropomi_raw)

    # Drop rows with ANY NaN (strict base)
    tropomi_base = tropomi_raw.dropna().copy()
    log(f"TROPOMI after dropping NaN: {len(tropomi_base):,} observations")

    # Encode primary_fuel_type AFTER dropna
    if not pd.api.types.is_integer_dtype(tropomi_base['primary_fuel_type']) and \
       not pd.api.types.is_float_dtype(tropomi_base['primary_fuel_type']):
        le_tmp = LabelEncoder()
        tropomi_base['primary_fuel_type'] = le_tmp.fit_transform(tropomi_base['primary_fuel_type'])
        log("Encoded 'primary_fuel_type' in TROPOMI (post-dropna).")

    # Get unique plant IDs from the strict base
    all_ids = tropomi_base['location'].dropna().unique().tolist()
    plants_for_all = plants[plants['Facility_ID'].isin(all_ids)].copy()
    log(f"TROPOMI contains {len(all_ids)} unique plants; metadata rows matched: {len(plants_for_all)}.")

    # Filter annual emissions to only include plants in TROPOMI
    annual_emissions_filtered = annual_emissions[annual_emissions['Facility ID'].isin(all_ids)].copy()
    log(f"Annual emissions filtered to {len(annual_emissions_filtered):,} records")

    # Identify interference zones BY YEAR
    log("\nIdentifying interference zones BY YEAR...")
    year_interfered_dict = identify_plants_in_interference_zones_by_year(
        plants_for_all, 
        annual_emissions_filtered,
        cities, 
        plant_subset_ids=all_ids
    )

    # Calculate total unique interfered plants across all years
    all_interfered_plants = set()
    for year_set in year_interfered_dict.values():
        all_interfered_plants.update(year_set)

    # Build datasets from the SAME strict base
    unfiltered_all = tropomi_base.copy()
    filtered_all = filter_data_by_year_interference(tropomi_base, year_interfered_dict)

    # Stats
    log("\nDataset statistics:")
    log(f"  Total unique plants: {len(all_ids)}")
    log(f"  Ever interfered plants (across all years): {len(all_interfered_plants)}")
    log(f"  Never interfered plants: {len(all_ids) - len(all_interfered_plants)}")
    log(f"  Original observations: {len(unfiltered_all):,}")
    log(f"  Year-filtered observations: {len(filtered_all):,}")
    log(f"  Removed by year-filtering: {len(unfiltered_all) - len(filtered_all):,}")

    # Year-by-year statistics
    log("\nYear-by-year interference statistics:")
    for year in sorted(year_interfered_dict.keys()):
        year_data = tropomi_base[tropomi_base['year'] == year]
        year_plants = year_data['location'].unique()
        year_interfered = year_interfered_dict[year]
        log(f"  Year {year}: {len(year_plants)} plants active, {len(year_interfered)} interfered ({len(year_interfered)/len(year_plants)*100:.1f}%)")

    # Save interference info
    info = {
        "summary": {
            "total_plants": len(all_ids),
            "ever_interfered_plants": len(all_interfered_plants),
            "never_interfered_plants": len(all_ids) - len(all_interfered_plants),
            "original_observations": int(len(unfiltered_all)),
            "filtered_observations": int(len(filtered_all)),
            "removed_observations": int(len(unfiltered_all) - len(filtered_all))
        },
        "year_specific": {}
    }
    
    for year, interfered_set in year_interfered_dict.items():
        info["year_specific"][str(year)] = {
            "interfered_count": len(interfered_set),
            "interfered_plant_ids": list(map(str, interfered_set))
        }
    
    with open(os.path.join(OUT_ROOT, 'interference_info_yearly_perm_importance.json'), 'w') as f:
        json.dump(info, f, indent=2)

    # Output dirs
    out_unfiltered = os.path.join(OUT_ROOT, 'unfiltered')
    out_filtered   = os.path.join(OUT_ROOT, 'filtered')
    os.makedirs(out_unfiltered, exist_ok=True)
    os.makedirs(out_filtered, exist_ok=True)

    # Run permutation importance on both datasets
    log("\n" + "="*80)
    log("RUNNING PERMUTATION IMPORTANCE ANALYSIS")
    log("="*80)
    
    results = {}
    
    # Unfiltered dataset with original model
    # log("\n--- Analyzing UNFILTERED dataset (all observations) ---")
    # results['unfiltered_all'] = run_perm_importance_for_dataset(
    #     'unfiltered_all', 
    #     unfiltered_all, 
    #     MODEL_PATH,  # Original model
    #     out_unfiltered, 
    #     device
    # )
    
    # Filtered dataset with filtered model (if available)
    log("\n--- Analyzing FILTERED dataset (year-specific interference removed) ---")
    # Try to use filtered model if it exists, otherwise use original
    filtered_model_path = MODEL_PATH_FILTERED if os.path.exists(MODEL_PATH_FILTERED) else MODEL_PATH
    if filtered_model_path == MODEL_PATH_FILTERED:
        log(f"Using year-filtered model: {MODEL_PATH_FILTERED}")
    else:
        log(f"Year-filtered model not found, using original: {MODEL_PATH}")
    
    results['filtered_all'] = run_perm_importance_for_dataset(
        'filtered_all_yearly', 
        filtered_all,
        filtered_model_path,
        out_filtered, 
        device
    )

    # Compare top features between datasets
    log("\n" + "="*80)
    log("FEATURE IMPORTANCE COMPARISON")
    log("="*80)
    
    # log("\nTop 5 features - UNFILTERED data:")
    # if 'top_5_features' in results['unfiltered_all']:
    #     for i, (feat, imp) in enumerate(zip(results['unfiltered_all']['top_5_features'], 
    #                                         results['unfiltered_all']['top_5_importances'])):
    #         log(f"  {i+1}. {feat}: {imp:.4f}")
    
    log("\nTop 5 features - FILTERED data:")
    if 'top_5_features' in results['filtered_all']:
        for i, (feat, imp) in enumerate(zip(results['filtered_all']['top_5_features'], 
                                            results['filtered_all']['top_5_importances'])):
            log(f"  {i+1}. {feat}: {imp:.4f}")

    # Save summary
    with open(os.path.join(OUT_ROOT, 'perm_importance_summary_yearly.json'), 'w') as f:
        json.dump(results, f, indent=2)

    # Create comparison plot
    # if 'top_5_features' in results['unfiltered_all'] and 'top_5_features' in results['filtered_all']:
    #     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
        
        # # Unfiltered
        # top_unfilt = results['unfiltered_all']
        # ax1.barh(range(5), top_unfilt['top_5_importances'][::-1], color='steelblue')
        # ax1.set_yticks(range(5))
        # ax1.set_yticklabels([f[:30]+'...' if len(f) > 30 else f for f in top_unfilt['top_5_features'][::-1]])
        # ax1.set_xlabel('Permutation Importance (AUC drop)')
        # ax1.set_title('Top 5 Features - Unfiltered Data')
        # ax1.grid(True, alpha=0.3, axis='x')
        
        # Filtered
        # top_filt = results['filtered_all']
        # ax2.barh(range(5), top_filt['top_5_importances'][::-1], color='green')
        # ax2.set_yticks(range(5))
        # ax2.set_yticklabels([f[:30]+'...' if len(f) > 30 else f for f in top_filt['top_5_features'][::-1]])
        # ax2.set_xlabel('Permutation Importance (AUC drop)')
        # ax2.set_title('Top 5 Features - Year-Filtered Data')
        # ax2.grid(True, alpha=0.3, axis='x')
        
        # plt.suptitle('Permutation Importance Comparison: Original vs Year-Filtered Data', fontsize=14, y=1.02)
        # plt.tight_layout()
        # plt.savefig(os.path.join(OUT_ROOT, 'feature_importance_comparison_yearly.png'), dpi=300, bbox_inches='tight')
        # plt.close()
        # log(f"Saved comparison plot: {os.path.join(OUT_ROOT, 'feature_importance_comparison_yearly.png')}")

    log("\n" + "="*80)
    log("ANALYSIS COMPLETE")
    log("="*80)
    log(f"\nAll results saved to: {OUT_ROOT}")
    log("Done.")

[2025-08-28 14:22:18] Loading plants: /net/fs06/d3/rzhuang/TROPOMI_US/data/facility_emissions_by_plant_comprehensive.csv
[2025-08-28 14:22:18] Loading annual emissions: /net/fs06/d3/rzhuang/TROPOMI_US/data/annual-emissions-facility-aggregation-2019-2024.csv
[2025-08-28 14:22:18] Loading cities: /net/fs06/d3/rzhuang/TROPOMI_world/data/worldcities.csv
[2025-08-28 14:22:18] Loading TROPOMI: /net/fs06/d3/rzhuang/TROPOMI_US/data/Run_20250623_203825/updated_tropomi_hourly_emissions_full_variables.csv
[2025-08-28 14:22:26] Extracting year from TROPOMI data...
[2025-08-28 14:22:26] Error parsing utc_time: time data "2022-07-27 20:47:56+00:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f%z", at position 72. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

# Load data
tropomi_combined_dropna = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/Run_20250623_203825/updated_tropomi_hourly_emissions_full_variables.csv').dropna()

# Replace with:
FEATURES = [
    'annual_nox_emission', 'surface_altitude', 'surface_altitude_precision',
    'surface_classification', 'surface_pressure', 'surface_albedo',
    'surface_albedo_nitrogendioxide_window', 'cloud_pressure_crb',
    'cloud_fraction_crb', 'cloud_albedo_crb', 'scene_albedo',
    'apparent_scene_pressure', 'snow_ice_flag', 'aerosol_index_354_388', 
    'scaled_small_pixel_variance', 'tropospheric_NO2_column_number_density', 
    'sensor_altitude', 'sensor_azimuth_angle', 'sensor_zenith_angle', 
    'solar_azimuth_angle', 'solar_zenith_angle', 'wind_speed', 't2m', 'tisr', 'tcwv',
    'no2_mean_radius', 'no2_std_radius', 'no2_frac_valid_radius', 'primary_fuel_type', 'NOx Mass (lbs)'
]

feature_names = FEATURES
print(f"Number of features to analyze: {len(FEATURES)}")

le = LabelEncoder()
tropomi_combined_dropna['primary_fuel_type'] = le.fit_transform(tropomi_combined_dropna['primary_fuel_type'])
# Extract X with the specified features
X = tropomi_combined_dropna[FEATURES].values.astype(np.float32)
y = tropomi_combined_dropna["plume_label"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ros = RandomOverSampler(random_state=42)
X_tr_bal, y_tr_bal = ros.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_bal)
X_te_s = scaler.transform(X_test)

# Model definition
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

# Load trained model (the one trained without nearby features)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(X_tr_s.shape[1]).to(device)
model.load_state_dict(torch.load(f'/net/fs06/d3/rzhuang/TROPOMI_US/model/best_0.0001_features_no_nearby.pt'))
model.eval()

# Calculate original AUC
test_ds = TensorDataset(torch.from_numpy(X_te_s), torch.from_numpy(y_test))
test_dl = DataLoader(test_ds, batch_size=32)

def calculate_auc(model, data_loader, y_true):
    probs = []
    with torch.no_grad():
        for xb, _ in data_loader:
            xb = xb.to(device)
            probs.extend(torch.sigmoid(model(xb)).cpu().numpy())
    return roc_auc_score(y_true, probs)

orig_auc = calculate_auc(model, test_dl, y_test)
print(f"\nOriginal AUC: {orig_auc:.4f}")

# Permutation importance
n_features = X_te_s.shape[1]
importances = np.zeros(n_features)
n_repeats = 5  # Number of times to repeat permutation

print("\nCalculating permutation importance...")
for i in tqdm(range(n_features)):
    scores = []
    for _ in range(n_repeats):
        # Create a copy of test data
        X_perm = X_te_s.copy()
        # Permute feature i
        np.random.shuffle(X_perm[:, i])
        # Create new dataloader with permuted data
        perm_ds = TensorDataset(torch.from_numpy(X_perm), torch.from_numpy(y_test))
        perm_dl = DataLoader(perm_ds, batch_size=32)
        # Calculate AUC with permuted feature
        perm_auc = calculate_auc(model, perm_dl, y_test)
        # Importance is the drop in AUC
        scores.append(orig_auc - perm_auc)
    importances[i] = np.mean(scores)

# Create results dataframe
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(importance_df.head(20))

# Save results
importance_df.to_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/Run_20250623_203825/feature_importance_permutation_no_nearby.csv', index=False)

# Visualize
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
top_features = importance_df.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Permutation Importance (AUC drop)')
plt.title('Top 20 Most Important Features (No Nearby Features)')
plt.tight_layout()
plt.savefig('/net/fs06/d3/rzhuang/TROPOMI_US/data/Run_20250623_203825/feature_importance_plot_no_nearby.png', dpi=300)
plt.close()

print(f"\nResults saved to feature_importance_permutation_no_nearby.csv")
print(f"Plot saved to feature_importance_plot_no_nearby.png")

Number of features to analyze: 30

Original AUC: 0.9169

Calculating permutation importance...


100%|██████████| 30/30 [06:30<00:00, 13.01s/it]



Top 20 Most Important Features:
                                   feature  importance
1                         surface_altitude    0.097149
0                      annual_nox_emission    0.084421
16                         sensor_altitude    0.072362
15  tropospheric_NO2_column_number_density    0.063895
2               surface_altitude_precision    0.063505
28                       primary_fuel_type    0.060776
3                   surface_classification    0.054234
29                          NOx Mass (lbs)    0.035435
6    surface_albedo_nitrogendioxide_window    0.034176
25                         no2_mean_radius    0.031009
26                          no2_std_radius    0.029781
5                           surface_albedo    0.023542
4                         surface_pressure    0.021942
10                            scene_albedo    0.013877
20                      solar_zenith_angle    0.013677
27                   no2_frac_valid_radius    0.012951
18                     sensor_ze